In [1]:
# Celda 1: Montar Google Drive e importar librerías
from google.colab import drive
drive.mount('/content/drive')

!pip install pyspark -q

# Celda 2: Inicializar Spark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, mean, round, isnan, count
from pyspark.sql.types import IntegerType, FloatType, StringType
import pandas as pd

spark = SparkSession.builder.appName("TitanicETL").getOrCreate()

# Celda 3: Cargar datos
# Opción 1: Desde Google Drive
df = spark.read.csv("/content/drive/MyDrive/titanic.csv", header=True, inferSchema=True)

# Opción 2: Desde URL directa (si tienes acceso)
# df = spark.read.csv("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv", header=True, inferSchema=True)

print(f"Registros cargados: {df.count()}")
df.printSchema()
df.show(5)

# Celda 4: Pipeline ETL completo
# 4.1 Limpieza de nulos
df_clean = df.dropna(subset=["Survived", "Pclass", "Sex", "Age"])

# 4.2 Imputar edad faltante (si aún hay nulos)
avg_age = df_clean.select(mean("Age")).collect()[0][0]
df_clean = df_clean.fillna({"Age": avg_age})

# 4.3 Eliminar columnas no necesarias
df_clean = df_clean.drop("PassengerId", "Ticket", "Cabin", "Name")

# 4.4 Convertir tipos de datos
df_clean = df_clean.withColumn("Age", col("Age").cast(IntegerType()))
df_clean = df_clean.withColumn("Fare", col("Fare").cast(FloatType()))

# 4.5 Crear variables derivadas
from pyspark.sql.functions import when, col, lit

df_final = df_clean.withColumn(
    "AgeGroup",
    when(col("Age") < 18, "Child")
    .when(col("Age") < 30, "Young_Adult")
    .when(col("Age") < 50, "Adult")
    .otherwise("Senior")
)

df_final = df_final.withColumn(
    "FamilySize",
    col("SibSp") + col("Parch") + 1
)

df_final = df_final.withColumn(
    "IsAlone",
    when(col("FamilySize") == 1, 1).otherwise(0)
)

# 4.6 Codificar variables categóricas
from pyspark.ml.feature import StringIndexer

indexer_sex = StringIndexer(inputCol="Sex", outputCol="SexIndex", handleInvalid="keep")
indexer_embarked = StringIndexer(inputCol="Embarked", outputCol="EmbarkedIndex", handleInvalid="keep")
indexer_agegroup = StringIndexer(inputCol="AgeGroup", outputCol="AgeGroupIndex", handleInvalid="keep")

df_final = indexer_sex.fit(df_final).transform(df_final)
df_final = indexer_embarked.fit(df_final).transform(df_final)
df_final = indexer_agegroup.fit(df_final).transform(df_final)

# Celda 5: Guardar datos procesados
df_final.write.mode("overwrite").parquet("/content/drive/MyDrive/titanic_clean.parquet")

# Celda 6: Verificar resultado
print("Datos finales:")
df_final.show(5)
df_final.printSchema()

# Celda 7: Estadísticas básicas
print(f"Total registros: {df_final.count()}")
print(f"Columnas: {df_final.columns}")
df_final.describe(["Age", "Fare", "SibSp", "Parch"]).show()

Mounted at /content/drive
Registros cargados: 891
root
 |-- PassengerId: integer (nullable = true)
 |-- Survived: integer (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- SibSp: integer (nullable = true)
 |-- Parch: integer (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: double (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     